# Pasos previos

## Imports y dependencias

In [40]:
pip install scikit-learn pandas numpy gensim nltk

Note: you may need to restart the kernel to use updated packages.


In [41]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.linear_model import SGDClassifier
from sklearn.mixture import GaussianMixture
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import FunctionTransformer

import nltk
nltk.download('stopwords')
nltk.download('punkt')
import pandas as pd
import re
from nltk.tokenize import TweetTokenizer, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer


[nltk_data] Downloading package stopwords to
[nltk_data]     /home/agustin/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/agustin/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


## Carga de archivo y preprocesamiento

In [42]:
tknzr =  TweetTokenizer(preserve_case=False)
stemmer = SnowballStemmer('spanish')

nltk.download("stopwords", quiet=True)
stop_es = set(stopwords.words("spanish"))

def preprocesar(text): # Preprocesamiento hecho
  texto = text.strip()
  texto = tknzr.tokenize(texto)
  texto = [stemmer.stem(t) for t in texto if t not in stop_es]
  texto = ' '.join(texto)
  patron = r'[^\w\sáéíóúüñáàèìòùäëïöü]'
  texto = re.sub(patron, '', texto)
  texto = ' '.join(texto.split())
  return texto

In [43]:
name = 'news_reducido.csv'

# Leer los datos en formato csv
data = pd.read_csv(name, on_bad_lines='skip')

# Nos quedamos con el texto (puedes quedarte con más información si quieres)
dataset = np.array([preprocesar(t) for t in data['text'].fillna('')])

## Datos utilizados

In [44]:
semilla1=42
semilla2=640
semilla3=5300742

semilla = semilla1

#Agrupamiento
num_clusters=4



## Validación cruzada estratificada

In [45]:
enc = OrdinalEncoder()
y = enc.fit_transform(np.reshape(data['category'], (-1,1))).reshape(-1) # DOCUMENTOS Y LE ASOCIO LA CATEGORIA

# Hacemos la partición train-test con Validacion cruzada estratificada
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=semilla)
#skf.get_n_splits(X, y)

# Agrupamiento

## Definición de los modelos

### Representación binaria

In [46]:
text_kmeans_binario = Pipeline([
    ('vect', CountVectorizer(binary=True)),
    ('clf', KMeans(n_clusters=num_clusters, random_state=semilla))
])

### Representación de frecuencias

In [47]:
text_kmeans_tf = Pipeline([
    ('vect', CountVectorizer(binary=False)),
    ('clf', KMeans(n_clusters=num_clusters, random_state=semilla))
])

### Representación TF-IDF

In [48]:
text_kmeans_tfidf = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', KMeans(n_clusters=num_clusters, random_state=semilla))
])

### Algoritmo de mezcla gaussiana

In [49]:
text_kmeans_tfidf_mezcla_gausiana = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('to_dense', FunctionTransformer(lambda x: x.toarray(), accept_sparse=True)),
    ('clf', GaussianMixture(n_components=num_clusters, random_state=semilla, covariance_type='spherical'))
])

## Ejecución de los modelos

### Representación binaria

In [50]:
text_kmeans = text_kmeans_binario

accuracies = np.zeros(5)

for i, (tra, tst) in enumerate(skf.split(X,y)):
    # Entrenamiento
    text_kmeans.fit(X[tra])
    # Test
    labels = text_kmeans.predict(X[tst])
    # Calculo de metricas
    acc = np.mean(labels == y[tst])
    print("Fold " + str(i+1) + ": " + str(acc))
    accuracies[i] = acc

# Tras el K-Fold, hay que mostrar la precision media obtenida (o cualquier otra metrica de interes, pero promediada)
avg_acc = np.average(accuracies)
print(f'Precision media = {avg_acc}')

Fold 1: 0.221
Fold 2: 0.221
Fold 3: 0.2345
Fold 4: 0.2835
Fold 5: 0.3015
Precision media = 0.25229999999999997


### Representación de frecuencias

In [51]:
text_kmeans = text_kmeans_tf

accuracies = np.zeros(5)

for i, (tra, tst) in enumerate(skf.split(X,y)):
    # Entrenamiento
    text_kmeans.fit(X[tra])
    # Test
    labels = text_kmeans.predict(X[tst])
    # Calculo de metricas
    acc = np.mean(labels == y[tst])
    print("Fold " + str(i+1) + ": " + str(acc))
    accuracies[i] = acc

# Tras el K-Fold, hay que mostrar la precision media obtenida (o cualquier otra metrica de interes, pero promediada)
avg_acc = np.average(accuracies)
print(f'Precision media = {avg_acc}')

Fold 1: 0.313
Fold 2: 0.2115
Fold 3: 0.182
Fold 4: 0.2015
Fold 5: 0.3015
Precision media = 0.24189999999999995


### Representación TF-IDF

In [52]:
text_kmeans = text_kmeans_tfidf

accuracies = np.zeros(5)

for i, (tra, tst) in enumerate(skf.split(X,y)):
    # Entrenamiento
    text_kmeans.fit(X[tra])
    # Test
    labels = text_kmeans.predict(X[tst])
    # Calculo de metricas
    acc = np.mean(labels == y[tst])
    print("Fold " + str(i+1) + ": " + str(acc))
    accuracies[i] = acc

# Tras el K-Fold, hay que mostrar la precision media obtenida (o cualquier otra metrica de interes, pero promediada)
avg_acc = np.average(accuracies)
print(f'Precision media = {avg_acc}')

Fold 1: 0.118
Fold 2: 0.109
Fold 3: 0.191
Fold 4: 0.0715
Fold 5: 0.109
Precision media = 0.1197


### Algoritmo de mezcla gaussiana


In [ ]:
text_kmeans = text_kmeans_tfidf_mezcla_gausiana

accuracies = np.zeros(5)

for i, (tra, tst) in enumerate(skf.split(X,y)):
    # Entrenamiento
    text_kmeans.fit(X[tra])
    # Test
    labels = text_kmeans.predict(X[tst])
    # Calculo de metricas
    acc = np.mean(labels == y[tst])
    print("Fold " + str(i+1) + ": " + str(acc))
    accuracies[i] = acc

# Tras el K-Fold, hay que mostrar la precision media obtenida (o cualquier otra metrica de interés, pero promediada)
avg_acc = np.average(accuracies)
print(f'Precision media = {avg_acc}')

Fold 1: 0.1295
